In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Load dataset (use parquet if created in convert_to_parquet.py, or fall back to csv)
try:
    df = pd.read_parquet("data/analysis_sample.parquet")
except Exception:
    df = pd.read_csv("powerbi/analysis_sample.csv")

# 2. Build target and isolate clean triage features
df['HIGH_BP_FLAG'] = np.where(
    (df['SYSTOLIC_BP'] >= 140) | (df['DIASTOLIC_BP'] >= 90), 1, 0
)

leakage_cols = [
    'SYSTOLIC_BP', 'DIASTOLIC_BP', 'DIAGNOSED_HYPERTENSION',
    'BP_CATEGORY', 'SAMPLE_ID'
]
X = df.drop(columns=leakage_cols + ['HIGH_BP_FLAG'])
y = df['HIGH_BP_FLAG']

# 3. Stratified split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Dataset split complete: Train shape = {X_train.shape}, Test shape = {X_test.shape}")

# 4. Fit DummyClassifier baseline (strategy='most_frequent')
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
dummy_preds = dummy.predict(X_test)
baseline_acc = accuracy_score(y_test, dummy_preds)

print(f"Majority class predicted: {dummy.classes_[np.argmax(dummy.class_prior_)]}")
print(f"DummyClassifier Baseline Accuracy: {baseline_acc:.4f} ({baseline_acc * 100:.2f}%)")

# 5. Define ColumnTransformer and Preprocessing Pipeline
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ]
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

# 6. Fit and transform training set to verify execution without errors
X_train_processed = pipeline.fit_transform(X_train)
X_test_processed = pipeline.transform(X_test)

print("Pipeline verified successfully.")
print(f"Processed training feature matrix shape: {X_train_processed.shape}")

Dataset split complete: Train shape = (126315, 8), Test shape = (31579, 8)
Majority class predicted: 1
DummyClassifier Baseline Accuracy: 0.5802 (58.02%)


C:\Users\ugochukwu.emenike\AppData\Local\Temp\ipykernel_15896\1729345702.py:46: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()


Pipeline verified successfully.
Processed training feature matrix shape: (126315, 56)
